# 01 Data profile and cleaning

Owner: Manavkrishna Manoj (Data Engineer)

This notebook profiles the raw data and shows why each cleaning rule is there. The rules themselves are in src/clean.py, so the notebook and build_matrix.py run the same code.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from build_matrix import load_raw
from clean import clean_raw, PRODUCT_CODE_PATTERN

pd.set_option("display.max_columns", 20)
df = load_raw()
print(df.shape)
# drop Customer ID from the preview so no real ID gets saved in the output
df.drop(columns="Customer ID").head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,United Kingdom


## Data profile

In [2]:
# the brief says column names can differ (Invoice/InvoiceNo, Price/UnitPrice), so check
print(df.columns.tolist())

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(),
})
profile

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,dtype,missing,missing_%,unique
Invoice,object,0,0.00,53628
StockCode,object,0,0.00,5305
Description,object,4382,0.41,5698
Quantity,int64,0,0.00,1057
InvoiceDate,datetime64[us],0,0.00,47635
Price,float64,0,0.00,2807
Customer ID,float64,243007,22.77,5942
Country,str,0,0.00,43


In [3]:
print("Rows:", f"{len(df):,}")
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Exact duplicate rows:", f"{df.duplicated().sum():,}")
profile.to_csv("../outputs/results/data_profile.csv")

Rows: 1,067,371
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Exact duplicate rows: 34,335


## Evidence for each rule

These are counts on the raw data, one rule at a time. The cleaning log further down applies the rules in order, so a row that breaks two rules only shows up once.

In [4]:
invoice = df["Invoice"].astype(str)
rules = pd.Series({
    "Cancellation invoices (start with C)": invoice.str.upper().str.startswith("C").sum(),
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "Price <= 0": (df["Price"] <= 0).sum(),
    "Missing Customer ID": df["Customer ID"].isna().sum(),
    "Missing Description": df["Description"].isna().sum(),
})
pd.DataFrame({"rows": rules, "% of raw": (rules / len(df) * 100).round(2)})

,rows,% of raw
Cancellation invoices (start with C),19494,1.83
Quantity <= 0,22950,2.15
Price <= 0,6207,0.58
Missing Customer ID,243007,22.77
Missing Description,4382,0.41


In [5]:
# negative price or quantity that isn't a cancellation
cols = ["Invoice", "StockCode", "Description", "Quantity", "Price"]
not_cancel = ~invoice.str.upper().str.startswith("C")
print("Negative price, not a cancellation:")
display(df.loc[(df["Price"] < 0) & not_cancel, cols].head(10))
print("Negative quantity, not a cancellation:")
display(df.loc[(df["Quantity"] < 0) & not_cancel, cols].head(10))

Negative price, not a cancellation:


,Invoice,StockCode,Description,Quantity,Price
179403,A506401,B,Adjust bad debt,1,-53594.36
276274,A516228,B,Adjust bad debt,1,-44031.79
403472,A528059,B,Adjust bad debt,1,-38925.87
825444,A563186,B,Adjust bad debt,1,-11062.06
825445,A563187,B,Adjust bad debt,1,-11062.06


Negative quantity, not a cancellation:


,Invoice,StockCode,Description,Quantity,Price
263,489464,21733,85123a mixed,-96,0.0
283,489463,71477,short,-240,0.0
284,489467,85123A,21733 mixed,-192,0.0
470,489521,21646,NaN,-50,0.0
3114,489655,20683,NaN,-44,0.0
3162,489660,35956,lost,-1043,0.0
3168,489663,35605A,damages,-117,0.0
4296,489806,18010,NaN,-770,0.0
4538,489820,21133,invcd as 84879?,-720,0.0
4566,489821,85049G,NaN,-240,0.0


### Non-product codes

The brief says a product code is 5 digits plus optional letters. Anything that fails that is listed here so we can check by eye that no real product gets thrown out.

In [6]:
code = df["StockCode"].astype(str).str.upper().str.strip()
not_product = ~code.str.fullmatch(PRODUCT_CODE_PATTERN)
print("Rows with a non-product code:", f"{not_product.sum():,}")
non_product_summary = (
    df[not_product]
    .assign(StockCode=code[not_product])
    .groupby("StockCode")
    .agg(rows=("StockCode", "size"), example=("Description", "first"))
    .sort_values("rows", ascending=False)
)
non_product_summary.head(40)

Rows with a non-product code: 5,935


,rows,example
StockCode,,
POST,2122,POSTAGE
DOT,1446,DOTCOM POSTAGE
M,1426,Manual
C2,282,CARRIAGE
D,177,Discount
S,104,SAMPLES
BANK CHARGES,102,Bank Charges
ADJUST,67,Adjustment by john on 26/01/2010 16
AMAZONFEE,43,AMAZON FEE


Decision: `PRODUCT_CODE_EXCEPTIONS` stays empty. Nearly everything that fails the pattern is postage, fees, vouchers, adjustments or test lines. SP1002 looks like a real product (a kid's chalkboard) but it only has 3 raw rows and the 10-customer filter would drop it anyway. PADS is an add-on line, so it stays out. The DCGS gift sets pass the pattern, but they have no customer ID so they never reach the matrix.

## Cleaning log

In [7]:
clean, cleaning_log, removed_codes = clean_raw(df)
cleaning_log

1. Remove cancellation invoices: removed 19,494, left 1,047,877
2. Remove quantity <= 0: removed 3,457, left 1,044,420
3. Remove price <= 0: removed 2,750, left 1,041,670
4. Remove non-product codes: removed 4,668, left 1,037,002
5. Remove missing Customer ID: removed 234,370, left 802,632
6. Remove exact duplicate lines: removed 26,055, left 776,577


,step,rule,rows_removed,rows_remaining
0,0,Raw dataset,0,1067371
1,1,Remove cancellation invoices,19494,1047877
2,2,Remove quantity <= 0,3457,1044420
3,3,Remove price <= 0,2750,1041670
4,4,Remove non-product codes,4668,1037002
5,5,Remove missing Customer ID,234370,802632
6,6,Remove exact duplicate lines,26055,776577


In [8]:
print("Clean rows:", f"{len(clean):,}", "| customers:", clean["Customer ID"].nunique(),
      "| products:", clean["StockCode"].nunique())
removed_codes.head(15)

Clean rows: 776,577 | customers: 5852 | products: 4619


StockCode
POST            1886
DOT             1436
M                881
C2               272
ADJUST            36
BANK CHARGES      35
GIFT_0001_20      26
GIFT_0001_30      24
PADS              17
GIFT_0001_10      14
TEST001            9
GIFT_0001_50       6
D                  5
GIFT_0001_40       5
AMAZONFEE          4
Name: count, dtype: int64

## Limits for the memo

Customer ID is missing on 243,007 of the 1,067,371 raw lines (22.77%). A customer by product matrix needs a customer, so those lines go, and guest purchases are invisible to the model. The other rules only look at one row at a time, so it's safe to run them before the split without leaking anything from the test period.